In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Machine Learning & Time-Series Models
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:

import pandas as pd

# Define the exact standard columns of the WFP dataset
column_names = [
    'date', 'admin1', 'admin2', 'market', 'latitude', 'longitude', 
    'category', 'commodity', 'unit', 'priceflag', 'pricetype', 
    'currency', 'price', 'usdprice'
]

# Load the CSV, skip the header and the #tag metadata row entirely (skip rows 0 and 1)
df = pd.read_csv('wfp_food_prices_uga (1).csv', header=None, skiprows=[0, 1], names=column_names, low_memory=False)

# Filter for your 4 selected crops
selected_crops = ['Maize (white)', 'Maize flour', 'Beans', 'Rice']
df_clean = df[df['commodity'].isin(selected_crops)].copy()

# Convert data types safely
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
df_clean['latitude'] = pd.to_numeric(df_clean['latitude'], errors='coerce')
df_clean['longitude'] = pd.to_numeric(df_clean['longitude'], errors='coerce')
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

# Drop missing values and sort chronologically
df_clean = df_clean.dropna(subset=['date', 'price']).copy()
df_clean = df_clean.sort_values(by=['market', 'commodity', 'date']).reset_index(drop=True)

print(f"Dataset successfully loaded and cleaned! Total rows: {len(df_clean)}")
display(df_clean.head(3))


Dataset successfully loaded and cleaned! Total rows: 4738


C:\Users\CATHY\AppData\Local\Temp\ipykernel_9964\1903029062.py:21: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')


,date,admin1,admin2,market,latitude,longitude,category,commodity,unit,priceflag,pricetype,currency,price,usdprice
0,2021-01-15,Adjumani,Adjumani,Adjumani (refugee settlement),3.337018,31.780471,pulses and nuts,Beans,KG,actual,Retail,UGX,3508.0,0.9576
1,2021-02-15,Adjumani,Adjumani,Adjumani (refugee settlement),3.337018,31.780471,pulses and nuts,Beans,KG,actual,Retail,UGX,3524.0,0.9710
2,2021-04-15,Adjumani,Adjumani,Adjumani (refugee settlement),3.337018,31.780471,pulses and nuts,Beans,KG,actual,Retail,UGX,4000.0,1.1225


In [14]:
# Create lag and rolling features per market and commodity
df_clean['lag_1'] = df_clean.groupby(['market', 'commodity'])['price'].shift(1)
df_clean['lag_3'] = df_clean.groupby(['market', 'commodity'])['price'].shift(3)
df_clean['rolling_mean_3'] = (
    df_clean.groupby(['market', 'commodity'])['price']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# Extract date components
df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year

# Drop rows with NaN values created by lag shifts
df_model = df_clean.dropna(subset=['lag_1', 'lag_3', 'rolling_mean_3']).copy()

# Encode categorical text values into numbers
markets = sorted(df_model['market'].unique().tolist())
commodities = sorted(df_model['commodity'].unique().tolist())

market2code = {m: i for i, m in enumerate(markets)}
commodity2code = {c: i for i, c in enumerate(commodities)}

df_model['market_encoded'] = df_model['market'].map(market2code)
df_model['commodity_encoded'] = df_model['commodity'].map(commodity2code)

print(f"Feature engineering complete. Modeling dataset rows: {len(df_model)}")

Feature engineering complete. Modeling dataset rows: 4357


In [15]:
features = [
    'market_encoded', 'commodity_encoded', 'lag_1', 'lag_3', 
    'rolling_mean_3', 'month', 'year', 'latitude', 'longitude'
]
X = df_model[features]
y = df_model['price']

# Chronological split (shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 3485
Testing samples: 872


In [16]:
print("Training XGBoost Regressor...")
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae = mean_absolute_error(y_test, xgb_preds)

print(f"XGBoost Test RMSE: {xgb_rmse:.2f} UGX")
print(f"XGBoost Test MAE: {xgb_mae:.2f} UGX")

# Save model and metadata artifacts for your web app prototype
joblib.dump(xgb_model, 'crop_price_xgboost_model.pkl')
joblib.dump({
    'markets': markets,
    'commodities': commodities,
    'market2code': market2code,
    'commodity2code': commodity2code
}, 'model_meta.pkl')
print("XGBoost artifacts saved successfully!")

Training XGBoost Regressor...
XGBoost Test RMSE: 228.32 UGX
XGBoost Test MAE: 163.83 UGX
XGBoost artifacts saved successfully!


In [17]:
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)

print(f"Random Forest Test RMSE: {rf_rmse:.2f} UGX")
print(f"Random Forest Test MAE: {rf_mae:.2f} UGX")

Training Random Forest Regressor...
Random Forest Test RMSE: 247.43 UGX
Random Forest Test MAE: 175.75 UGX


In [18]:
target_market = 'Owino'
target_crop = 'Maize (white)'

ts_subset = df_clean[(df_clean['market'] == target_market) & (df_clean['commodity'] == target_crop)].dropna(subset=['price'])
ts_subset = ts_subset.sort_values('date')
ts_series = ts_subset.set_index('date')['price'].resample('ME').mean().interpolate()

sarima_train = ts_series[:-12]
sarima_test = ts_series[-12:]

print(f"Fitting SARIMA for {target_crop} in {target_market}...")
sarima_model = SARIMAX(
    sarima_train, 
    order=(1, 1, 1), 
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False, 
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

forecast_obj = sarima_results.get_forecast(steps=12)
sarima_forecast = forecast_obj.predicted_mean

sarima_rmse = np.sqrt(np.mean((sarima_test.values - sarima_forecast.values) ** 2))
sarima_mae = np.mean(np.abs(sarima_test.values - sarima_forecast.values))

print(f"SARIMA Test RMSE: {sarima_rmse:.2f} UGX")
print(f"SARIMA Test MAE: {sarima_mae:.2f} UGX")

Fitting SARIMA for Maize (white) in Owino...
SARIMA Test RMSE: 1156.91 UGX
SARIMA Test MAE: 1084.63 UGX


In [19]:
comparison_df = pd.DataFrame({
    'Model Architecture': ['XGBoost Regressor', 'Random Forest Regressor', 'SARIMA (Owino Maize)'],
    'Test RMSE (UGX)': [round(xgb_rmse, 2), round(rf_rmse, 2), round(sarima_rmse, 2)],
    'Test MAE (UGX)': [round(xgb_mae, 2), round(rf_mae, 2), round(sarima_mae, 2)]
})

print("--- FINAL MODEL PERFORMANCE COMPARISON ---")
display(comparison_df)

--- FINAL MODEL PERFORMANCE COMPARISON ---


,Model Architecture,Test RMSE (UGX),Test MAE (UGX)
0,XGBoost Regressor,228.32,163.83
1,Random Forest Regressor,247.43,175.75
2,SARIMA (Owino Maize),1156.91,1084.63


In [21]:
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.title("Ugandan Crop Price Prediction System")
st.write("Final-Year Project Prototype: Forecasting staple food prices using Machine Learning.")

# Load saved model and metadata
model = joblib.load('crop_price_xgboost_model.pkl')
meta = joblib.load('model_meta.pkl')

markets = meta['markets']
commodities = meta['commodities']
market2code = meta['model_meta.pkl'] if 'model_meta.pkl' in meta else meta['market2code'] # fallback check

# Safely extract dictionaries
market2code = meta['market2code']
commodity2code = meta['commodity2code']

# Sidebar inputs
st.sidebar.header("Prediction Parameters")
selected_market = st.sidebar.selectbox("Select Market", markets)
selected_commodity = st.sidebar.selectbox("Select Commodity", commodities)
lag_1_price = st.sidebar.number_input("Last Month Price (UGX)", value=1500.0)
lag_3_price = st.sidebar.number_input("Price 3 Months Ago (UGX)", value=1450.0)
rolling_mean = st.sidebar.number_input("3-Month Rolling Average (UGX)", value=1475.0)
month = st.sidebar.slider("Month", 1, 12, 6)
year = st.sidebar.slider("Year", 2026, 2030, 2026)

if st.button("Predict Future Price"):
    m_enc = market2code[selected_market]
    c_enc = commodity2code[selected_commodity]
    
    # Default lat/long reference point
    lat, lon = 0.3476, 32.5825
    
    input_data = pd.DataFrame([[
        m_enc, c_enc, lag_1_price, lag_3_price, 
        rolling_mean, month, year, lat, lon
    ]], columns=[
        'market_encoded', 'commodity_encoded', 'lag_1', 'lag_3', 
        'rolling_mean_3', 'month', 'year', 'latitude', 'longitude'
    ])
    
    prediction = model.predict(input_data)[0]
    st.success(f"Estimated Price for {selected_commodity} in {selected_market}: **{prediction:,.2f} UGX / KG**")
"""

# Open with encoding='utf-8' to prevent Windows cp1252 emoji/Unicode errors
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("Streamlit prototype 'app.py' generated successfully! Run 'streamlit run app.py' in your terminal.")

Streamlit prototype 'app.py' generated successfully! Run 'streamlit run app.py' in your terminal.


In [22]:
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="Ugandan Crop Price Prediction System", page_icon="🌾", layout="centered")

# Initialize session state for login
if 'logged_in' not in st.session_state:
    st.session_state['logged_in'] = False

# Load saved model and metadata
@st.cache_resource
def load_artifacts():
    model = joblib.load('crop_price_xgboost_model.pkl')
    meta = joblib.load('model_meta.pkl')
    return model, meta['markets'], meta['commodities'], meta['market2code'], meta['commodity2code']

model, markets, commodities, market2code, commodity2code = load_artifacts()

# --- AUTHENTICATION SCREEN ---
if not st.session_state['logged_in']:
    st.title("🌾 Ugandan Crop Price Prediction System")
    st.subheader("Farmer Portal Login")
    st.write("Please log in with your credentials or farmer ID to access market forecasts.")
    
    with st.form("login_form"):
        username = st.text_input("Farmer ID / Username", placeholder="e.g. farmer_001")
        password = st.text_input("Access PIN / Password", type="password", placeholder="e.g. 1234")
        login_btn = st.form_submit_button("Log In")
        
        if login_btn:
            # Simple validation logic (you can connect this to a database later for your project)
            if username.strip() != "" and password.strip() != "":
                st.session_state['logged_in'] = True
                st.session_state['username'] = username
                st.success("Login successful! Redirecting...")
                st.rerun()
            else:
                st.error("Please enter a valid Farmer ID and PIN.")

# --- MAIN PREDICTION DASHBOARD (AFTER LOGIN) ---
else:
    st.title("🇺🇬 Ugandan Crop Price Prediction Dashboard")
    st.success(f"Welcome, Farmer **{st.session_state['username']}**! Ready to forecast market prices.")
    
    if st.button("Log Out"):
        st.session_state['logged_in'] = False
        st.rerun()

    st.markdown("---")

    # Sidebar inputs
    st.sidebar.header("Prediction Parameters")
    selected_market = st.sidebar.selectbox("Select Market", markets)
    selected_commodity = st.sidebar.selectbox("Select Commodity", commodities)
    
    lag_1_price = st.sidebar.number_input("Last Month Price (UGX)", value=1500.0, step=50.0)
    lag_3_price = st.sidebar.number_input("Price 3 Months Ago (UGX)", value=1450.0, step=50.0)
    rolling_mean = st.sidebar.number_input("3-Month Rolling Average (UGX)", value=1475.0, step=50.0)
    
    month = st.sidebar.slider("Forecast Month", 1, 12, 6)
    year = st.sidebar.slider("Forecast Year", 2026, 2030, 2026)

    if st.button("Predict Future Price", type="primary"):
        m_enc = market2code[selected_market]
        c_enc = commodity2code[selected_commodity]
        
        # Default lat/long reference point
        lat, lon = 0.3476, 32.5825
        
        input_data = pd.DataFrame([[
            m_enc, c_enc, lag_1_price, lag_3_price, 
            rolling_mean, month, year, lat, lon
        ]], columns=[
            'market_encoded', 'commodity_encoded', 'lag_1', 'lag_3', 
            'rolling_mean_3', 'month', 'year', 'latitude', 'longitude'
        ])
        
        prediction = model.predict(input_data)[0]
        
        st.balloons()
        st.markdown(f"### 📊 Forecast Results")
        st.info(f"Estimated Price for **{selected_commodity}** in **{selected_market}** market:")
        st.success(f"**{prediction:,.2f} UGX / KG**")
"""

# Write with utf-8 encoding to avoid Windows cp1252 errors
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("Streamlit prototype 'app.py' with farmer login generated successfully!")

Streamlit prototype 'app.py' with farmer login generated successfully!
